In [ ]:
%%bash
# Run this cell first to prepare for the rest
apt install mafft emboss iqtree
git clone https://github.com/structbioinfo/BCH302
cp -r BCH302/* .

# **Practical: There has been an outbreak of "disease X" on campus**

---

**Author:** Dr Olivier Sheik Amamuddy $^{1}$

$^{1}$ Computational Molecular Biology Research Group (CoMBRG), 

Biochemistry, Microbiology and Bioinformatics Deptartment, Rhodes University

**Course:** BCH302 Bioinformatics

---

**Introduction**

- You and your classmates have been getting flu-like symptoms over the last few days 
- Oral swabs containing your samples were sent to a sequencing facility
- The genetic material obtained from the swab was purified, sequenced and the results were sent back to you
- One fragment was obtained from your swab, which we will now investigate

**Aim**

You need to acquire a maximum of useful information about the sequence that can be used to assess and understand the gravity of the outbreak, and find out how we can intervene, as bioinformaticists. We'll be using freely available tools and databases.

**Main objectives**

1. Determine what we are dealing with
   - Get a maximum of useful information (metadata) using computation, and using various public databases
2. Determine if everyone infected with the same pathogen
   - Multiple sequence alignment
   - Phylogenetic tree calculation
3. Evaluate an inhibitory compound against a reconstructed, credible 3D model of your target protein

---

**Overall Workflow**

<div style="background:rgb(184, 177, 175);color:white;padding:0.9rem;font-weight:bold;border:2px solid grey;border-radius:5px">1. Sequence Characterisation → 2. Sequence Alignments → 3. Conservation/Phylogenetic inference → 4. Protein Structure inference → 5. Drug discovery
</div>

---

## Preamble

If we are dealing with a completely new disease, we therefore assume have no pre-existing, or complete biological information about it from  online databases. We resort to analyse and annotate the initial sequence we obtained in order to try to understand sequence data ourselves using various tools and databases. We study its spread and attempt to target it using drug discovery techniques.

## **Part I - Individual Sequence Analysis**

Replace "STUDENT_ID" with your student ID in the cell below, and press "Ctrl+Enter" (making sure that cell is clicked on)

In [ ]:
%%bash
studentid="STUDENT_ID"
cp patient_sequences/patient_${studentid}.fasta patient_sequence.fasta && echo "DNA sequence successfully obtained" || echo "Input a correct student ID"

---

### 1.1. Determine possible proteins encoded by the sequence using ORF scanning

We first see if it codes for a protein, by scanning the sequence for all possible reading frames using the **standard genetic code**

In [ ]:
!sixpack -sequence patient_sequence.fasta -outseq orfs.fasta -html -outfile orfs.sixpack -table 0

<a href="orfs.sixpack" target="_blank">Visualize the ORF predictions produced from each reading frame</a>

[Click here to examine predicted protein sequences from each ORF in FASTA format](orfs.fasta)

<div style="background:rgba(218, 238, 132, 0.13);color:black;border:2px solid grey;border-radius:5px">

  <div style="background:rgb(218, 238, 132);color:black;padding:0.9rem;font-weight:bold">Question 1.1</div>

- How many ORFs did you predict, from each reading frame (F1, F2, F3, F4, F5 and F6)?
  - Hint: scroll down the output file
- Because proteins tend to be of a certain length, picking the longest ORFs will increase our chances of finding the actual protein coded by a coding sequence (CDS).
  - Looking at the peptide sequences inferred from the ORFs, which are the longest ones?
  - The start and end codon may not always be found in a sequence fragment. Why do you think that could happen?
- Which amino acid is encoded by a stop codon?
- The **standard genetic** code is used by default "(-table 0)". Change the genetic code from 0 to 2 (which is for vertebrate mitochondria) and re-run the `sixpack` command above
  - Examine the two output files produced by sixpack. What happened to the ORF that you got earlier?
- **When you're done with the previous question, change the table option back to "-table 0", and re-run the cell**

</div>

---

### 1.2. Studying evolutionary signals present within the nucleotide sequence using a dot plot

Let's align the **nucleotide sequence against itself**, and show the result as a **dot plot**. Let's see whether patterns emerge.

In the dot plot, a single black dot represents a **"match"** found at that position. When they are close, they form lines, which may go in various directions, which may hint at the evolutionary history of the sequence.

In [ ]:
!dotmatcher -asequence patient_sequence.fasta -bsequence patient_sequence.fasta -graph svg -windowsize 10 -threshold 23 -auto

Created dotmatcher.svg


<a href="dotmatcher.svg" target="_blank"> Click to examine the self dot plot produced from the DNA sequence</a>

<div style="background:rgba(218, 238, 132, 0.13);color:black;border:2px solid grey;border-radius:5px">

  <div style="background:rgb(218, 238, 132);color:black;padding:0.9rem;font-weight:bold">Question 1.2</div>

- Run dotmatcher with the default threshold score (threshold 23).
  - What is the middle diagonal line showing, and why?
  - Remember how this plot looks like, roughly.
- Decrease the threshold score to 13, and re-run the cell
  - How does it differ from the previous plot?
- Increase the threshold score to 30.
  - How does it differ from the previous plot?
- Restore the default threshold value to 23, then increase the window size from 10 to 50.
  - What changed?
</div>

---

### 1.3. Extract the protein sequence from the longest ORF and verify its sequence

Let's extract the longest ORF from the nucleotide sequence, that we're assuming to be coding. 

Run the following command, and increase "minsize" to sift out the smaller predicted peptides, until you get the longest one. Note that the keyword "minsize" is the nucleotide sequence length.

In [ ]:
!getorf -sequence patient_sequence.fasta -outseq patient_protein_sequence.fasta -minsize 30

Find and extract open reading frames (ORFs)


<a href="patient_protein_sequence.fasta" target="_blank"> Click to examine what protein you have extracted, and repeat with a higher number if needed.</a>

You may get cases where multiple valid ORFs, or ORF fragments, are found depending on the characteristics of the sequence. In that case, copy your longest predicted protein and paste it in the BLAST tool, and search a protein database (using [blastp](https://blast.ncbi.nlm.nih.gov/Blast.cgi)) to see if it there's any sequence that resembles yours. If you find hits with low E-value and high coverage, you can be confident in your ORF.

<div style="background:rgba(218, 238, 132, 0.13);color:black;border:2px solid grey;border-radius:5px">

  <div style="background:rgb(218, 238, 132, 1);color:black;padding:0.9rem;font-weight:bold">Question 1.3</div>

- What value of the minsize parameter did you choose? 
- Look at the pair of numbers in between the square brackets from the line of text that's just above the sequence. What is the indicated range of the nucleotide sequence that contained the ORF? 
</div>

---

### 1.4. Studying evolutionary signals present within the longest ORF's protein sequence using a dot plot

This time, let's align the **protein sequence** obtained from the longest ORF against itself, as a grid and see whether different patterns emerge on the dot plot.

In [ ]:
!dotmatcher -asequence patient_protein_sequence.fasta -bsequence patient_protein_sequence.fasta -windowsize 10 -threshold 23 -graph svg -auto

Created dotmatcher.svg


<a href="dotmatcher.svg" target="_blank"> Click to examine the self dot plot produced from the longest ORF's protein sequence</a>

<div style="background:rgba(218, 238, 132, 0.13);color:black;border:2px solid grey;border-radius:5px">

  <div style="background:rgb(218, 238, 132, 1);color:black;padding:0.9rem;font-weight:bold">Question 1.4</div>

- Run dotmatcher with the default threshold score (value=23) and windowsize (value=10). 
  - Compared to when the ORF nucleotide sequence was used for the dot plot, what is immediately apparent in this plot, when the protein sequence is used?
- Increase the windowsize from 23 to 50.
  - What changed? Why do you think that happened?
</div>

---

### 1.5. Finding any pre-existing information about sequence using homology search

Let's use **NCBI BLAST** to get more details about the sequence. We could use any of these four variations of the BLAST search tool:
- <a href="https://blast.ncbi.nlm.nih.gov/Blast.cgi?PROGRAM=tblastn&PAGE_TYPE=BlastSearch&LINK_LOC=blasthome">**blastn**</a>: uses a query DNA sequence to search against a database of DNA sequences 
  - Works whether the DNA codes for a protein or not
- <a href="https://blast.ncbi.nlm.nih.gov/Blast.cgi?PROGRAM=blastp&PAGE_TYPE=BlastSearch&LINK_LOC=blasthome">**blastp**</a>: uses a query protein sequence to search a database of protein sequences 
  - We could use the protein sequence that we predicted from the longest ORF
  - Note: this assumes that the DNA codes for a protein
- <a href="https://blast.ncbi.nlm.nih.gov/Blast.cgi?PROGRAM=blastx&PAGE_TYPE=BlastSearch&LINK_LOC=blasthome">**blastx**</a>: Uses a 6-frame translations of our query DNA sequence to search a protein database
- <a href="https://blast.ncbi.nlm.nih.gov/Blast.cgi?PROGRAM=tblastn&PAGE_TYPE=BlastSearch&LINK_LOC=blasthome">**tblastn**</a>: Uses a query protein sequence to search a database of translated nucleotides

<div style="background:rgba(218, 238, 132, 0.13);color:black;border:2px solid grey;border-radius:5px">

  <div style="background:rgb(218, 238, 132, 1);color:black;padding:0.9rem;font-weight:bold">Question 1.5</div>

- <a href="./patient_sequence.fasta" target="_blank" >Copy and paste your original DNA sequence into **blastn**. Run blastn.</a>
- <a href="./sars-cov-2-plpro_protein.fasta" target="_blank" >Copy and paste your translated longest ORF into **blastp**. Run blastp.</a>
- The results that you get from a BLAST search are sorted by the **E-value**. The lower the value, the more likely the match did not happen by chance.
  - Based on on this fact and on your findings, what are:
    - (a) the common name, (b) the genus and (c) the species of the organism that "infected" you? (Hint: Explore the "Taxonomy" tab). This information will be used later.
    - Why do you think that knowing the biological origins of the sequence would be helpful?
    - Do "blastp" and "blastn" give you the same "hits" (i.e. matched sequences)? If they differ, why do you think that happens? (Hint: Think about the sequences available from the databases')
</div>

---

### 1.5. Retrieving functional information about our protein sequence using the UniProt database

Go to **[UniProt](https://www.uniprot.org)**, click on the BLAST link at the top of the page, and paste your translated longest ORF in to the textbox, which expects a sequence in FASTA format. Instead of using BLAST, which is computationally demanding in this case, you may see the option "View the matching sequence in **UniParc**", which uses the pasted string to search the UniParc database instead. This will only work if there is an exact match of your query sequence in its database. 

If the match isn't found, click on the link to run the BLAST search.

<div style="background:rgba(218, 238, 132, 0.13);color:black;border:2px solid grey;border-radius:5px">

  <div style="background:rgb(218, 238, 132, 1);color:black;padding:0.9rem;font-weight:bold">Question 1.5</div>

- How many cross-references (matches from various databases) were associated to your sequence?
- Which organism does the matched sequence belong to?
- Click on the top identifier from the table of cross-references. This sends you to the linked page
  - What's the name of the protein?
  - What's the function of the protein?
  - What are the molecular functions that are assigned to the protein? The term **"Automatic annotation"** means that the predictions were done computationally, and have not yet been reviewed by a person.
  - Are there any PTMs (Post-translational modifications) are associated for the protein sequence? What are they?
</div>

---

## **Part II - Investigating the spread of the disease**

### 2.1. Multiple Sequence alignment

Combine all the sequence FASTA files collected from the outbreak by pasting them together (one file after another)

In [67]:
# Prepare a multi-FASTA file by combining all the patient sequence files
!cat patient_sequences/patient_*.fasta > patient_sequences/combined_nt_sequences.fasta

Align the combined sequences using **multiple sequence alignment**

In [ ]:
!mafft --auto patient_sequences/combined_nt_sequences.fasta > patient_sequences/combined_nt_sequences_aln.fasta 

Visualize the sequence alignment using Jalview, 

[Click to open alignment file. Select all characters and copy the text](patient_sequences/combined_nt_sequences_aln.fasta)

[Click to open JalView web interface](https://www.jalview.org/jalview-js/jalviewjs/)

- Steps to input your aligned sequences to the web interface: 
  - Click on File → input alignment → from text box → Paste your copied sequences

<div style="background:rgba(218, 238, 132, 0.13);color:black;border:2px solid grey;border-radius:5px">

  <div style="background:rgb(218, 238, 132, 1);color:black;padding:0.9rem;font-weight:bold">Question 2.1</div>

- Compare the unaligned sequences against the aligned ones
  - what's a major difference you find between them?
- The **consensus sequence** section displays the most frequent nucleotide encountered at each given position in a multiple sequence alignment.
  - What is the first nucleotide in the first position of the alignment, and what is it percentage observation across all characters at that position?
- Below the consensus section, you'll see the **Occupancy** section, which is computed by the proportion of non-gap characters at a given aligned position.
  - Which nucleotide position contains less than 100% occupancy?
</div>

**Note:** Please close the JalView tab when you are done with this section, as it may interfere with your other tabs

---

### 2.2. Phylogenetic tree calculation - estimating evolutionary relatedness

Let's calculate an **unrooted tree** from all the students' nucleotide sequences by running the following code. The phylogenetic tree assumes that each of the aligned positions is **"homologous"**, meaning that they descend from a common evolutionary path. If the aligned parts are not homologous, the tree would be wrong.

As a side note, **all phylogenetic trees are models**. To increase the level of trust in these models, external supporting evidence is required.

In [ ]:
# Build tree
!iqtree3 -s patient_sequences/combined_nt_sequences_aln.fasta  -B 1000 -alrt 1000 -redo -m MFP

Run the following to examine the tree file that was produced. 

In [ ]:
!cat patient_sequences/combined_nt_sequences_aln.fasta.treefile

<div style="background:rgba(218, 238, 132, 0.13);color:black;border:2px solid grey;border-radius:5px">

  <div style="background:rgb(218, 238, 132, 1);color:black;padding:0.9rem;font-weight:bold">Question 2.2</div>

- Copy the output tree above and upload it to the [iTOL server](https://itol.embl.de/upload.cgi) for tree visualization. Paste your sequence in the input text box. Click "Upload", and then on "Unrooted". You'll see that the tree structure looks circular, and it seems that it's divided in two. Click on the middle **"branch"**, scroll to "Tree structure", and click on "Root the tree at midpoint". It will seem like nothing happened, but now click on the "rectangular" tab and you'll see that a root has been placed, inferred from the midpoint of the data. The length of the branches indicate the number of mutations that were observed, and each "bifurcation" represents a group (**"clade"**). The little branch at the left is the root of the tree; the outermost tips are the **leaves**; and the internal crossing points are the **internal nodes**, which each indicate a hypothetical source.
  - If you cut the tree close to the root, how many clades that you can see? Ask a couple of classmates from the other clade which organism they were infected with.
  - Now, ask the classmate who is just next to you in the tree which organism he/she identified using BLAST
  - Now that you've got these pieces of evidence, can you investigate the sequence of events that lead to your "infections"?
  - Click on "Advanced". At the "Bootstraps / metadata" section, click on "Display", then set "Data source" to be "bootstrap0" in order to display the bootrap values. Switch the Legend from "Off" to "On". The higher the bootstrap percentage, the higher the confidence in the placement of that "clade". Export the file to your Downloads folder.
  - Compare your tree with what your classmate obtained. Are they identical? Hint: Look at the pairs, larger groups, and bootstrap values. The orientation of the leaf nodes is meaningless.
</div>

Note: Another way to root the tree would be to search for an **outgroup** sequence, which is a sequence that is homologous to the remaining sequence, but more distant from them. The idea is that that sequence will be more different to the remaining sequences, leading to rest of the sequences to be relatively more similar. This will retain the important branching patterns, while now placing a root between the outgroup and the rest.

---

## **Part III - Targeting its structure for drug discovery**

We have extensively examined our longest open reading frame using various tech techniques. Using BLAST, we know that homologs of our protein already exists. We will now proceed to determine a possible structure from our protein sequence using conventional (homology modelling) and artificial intelligence-based techniques. Note that I said "a possible structure". Proteins are **inherently dynamic**, and they will most likely adopt multiple valid conformations.

### 3.1. Protein homology modelling using the SWISS-MODEL server

A homo model can be built in four steps, namely: (i) modelling template identification, (ii) alignment of the target sequence against the template sequence, (iii) model construction, and (iv) quality evaluation. Because **a template can be specified** in homology modeling, one can model and study very specific states of proteins.  For example, one could compare a ligand-bound protein against its apo (ligand-unbound) state.

Let's determine a 3D model of our predicted protein sequence. [Click here and copy the protein sequence from the longest ORF](patient_protein_sequence.fasta)

Go to the [SWISS-MODEL](https://swissmodel.expasy.org/interactive) server. Paste your predicted protein sequence into the input textbox of the web server. Use your student ID as "Project title". Then click on "Search for templates". It may take a around 15 minutes.

<div style="background:rgba(218, 238, 132, 0.13);color:black;border:2px solid grey;border-radius:5px">

  <div style="background:rgb(218, 238, 132, 1);color:black;padding:0.9rem;font-weight:bold">Question 3.1</div>

- By searching for a modeling template, you are able to model specific states of the protein. The templates are sorted according to a template score. 
  - Examine the **first template** in the table that's returned.
    - The **coverage** is the amount of the target sequence that is matched by the template sequence. What is it's value?
    - What's the **percentage identity** for the alignment between the target sequence and the template sequence (i.e the amount of identical residues)?
    - It is possible that the protein that we need to model is a monomer, a dimer or multimer (**oligomeric state**).
      - What is the oligomeric state state of the template protein?
    - We ideally want template structures of high **resolution** (i.e. ideally < 2.5 Å), in order to build a model of high-quality.
      - What is the resolution of that template structure?
    - As we will attempt to evaluate the docking of a drug to the protein, we will look for a template which already has a bound ligand.
      - Is there any ligand (small molecule) bound to the template protein?
    - What do you think will happen to the rest of the target protein that didn't match the template?
  - Examine the table and look for a high quality template that has a small molecule (not metal ions, or crystallisation buffer). Select it.
    - Note its coverage, identity, resolution, and oligomeric state.
    - Go to the "Alignment" tab and have a look at the sequence alignment. 
      - The sequences are interleaved (read from left to right, then go vertically)
      - What are the starting and ending residue positions (from the template, and the target)
  - Click on "Build Models" to proceed to modelling.
    - Examine the structure ont the right panel on the "Models" tab section. 
      - Toggle the **secondary structure** visualisation. Note the regions and the types of secondary structures that are of low confidence.
        - Hint: Low confidence regions are in red, while high confidence ones are in blue
        - Hint: Secondary structures (e.g. alpha helix, beta strand, loops, coils, etc)
  - Download the 3D homology model to the "Downloads" folder on your local machine
</div>

[Further short videos on SWISS-MODEL](https://www.youtube.com/playlist?list=PLoCxWrRWjqB3A1Jhs_HSbED9Z3ViH--bI)

---

### 3.2. Ab initio protein structure determination using the Alphafold Server

Alphafold is an **artificial neural network** trained using protein structures from the PDB. It **does not require a template** to model the structure of a protein sequence, and can also be **used to model a structure that has never been seen before**. 

Despite being a **black box algorithm** that takes an input sequence and outputs a protein structure, it has performed exceedingly well in predicting protein structures in [CASP competitions](https://www.nature.com/articles/d41586-020-03348-4). Most importantly, it reports its level of confidence as three metrics: **[pLDDT](https://www.ebi.ac.uk/training/online/courses/alphafold/inputs-and-outputs/evaluating-alphafolds-predicted-structures-using-confidence-scores/plddt-understanding-local-confidence/)**, **[pTM](https://www.ebi.ac.uk/training/online/courses/alphafold/inputs-and-outputs/evaluating-alphafolds-predicted-structures-using-confidence-scores/confidence-scores-in-alphafold-multimer/)** and **[PAE](https://www.ebi.ac.uk/training/online/courses/alphafold/inputs-and-outputs/evaluating-alphafolds-predicted-structures-using-confidence-scores/pae-a-measure-of-global-confidence-in-alphafold-predictions/)**. 

While the approach can be accurate, it can also produce wrong models, by **hallucinating**, which is why the structures need to be considered alongside the quality metrics. 

In this section, we will get a chance to see how it compares against the homology modelling approach.

[Click here and copy the protein sequence from the longest ORF](patient_protein_sequence.fasta)

Go to the [Alphafold Server](https://alphafoldserver.com). Paste your predicted protein sequence into the input textbox of the web server, and then click on "Continue and preview job". Use your student ID to label the job name. Then click on "Confirm and submit job"

<div style="background:rgba(218, 238, 132, 0.13);color:black;border:2px solid grey;border-radius:5px">

  <div style="background:rgb(218, 238, 132, 1);color:black;padding:0.9rem;font-weight:bold">Question 3.2</div>

- Scroll down to the results section and click on your completed job.
  - The **predicted local distance difference test (pLDDT)** is a residue-wise (local) measure of confidence 
    - What type of secondary structures tend to be represented in the low confidence regions?
  - The **Predicted Alignment Error (PAE)** indicates AlphaFold's confidence in the relative placement of residue pair. The lower the error value (darker green), the better. Well-packed domains will appear as contiguous low-PAE blocks, indicating the confidence in their placement.
    - Hover your mouse over graph on the right heat map, and select the blocks of low PAE.
- Compare your homology model to that inferred by AlphaFold, by **aligning their 3D structure** using **PyMOL**
  - Open PyMOL, and open each of the structures
    - If PyMOL is not installed, [Click here](https://pymol.org/#download) to download the "Windows EXE Installer". Follow the instructions for installing for the user (you)
    - Move your mouse to the name of the AlphaFold structure in the right panel, and click on "A" -> "Align" -> "all to this"
    - What is similar and what is different, structurally? (e.g. shape, length, arrangement, discordance in secondary structures)
    - Do you think that the **ligand-bound pocket conformation** present in the template-derived model is also present in the *ab initio* model?


</div>

---

### 3.3. Drug discovery using the CB-Dock 2 server

Now that we have the 3D structure of your protein, let's see if it can be targeted using **blind docking (BD)**, and hopefully inhibited by a small molecule. BD is used when the exact docking site from the protein is not known. The ligand (small molecule) is scanned across the entire protein's surface in the hope of finding low energy conformations (favorable, non-covalent interactions).

Search for the compound "GRL0617" from the [PubChem database](https://pubchem.ncbi.nlm.nih.gov). Click on the result, and click on "Download" on the top right corner. Select the "3D Conformer" in SDF format. Save it in your "Downloads" folder on your PC.

[Click here to go to the CB-Dock 2 server](https://cadd.labshare.cn/cb-dock2/)

<div style="background:rgba(218, 238, 132, 0.13);color:black;border:2px solid grey;border-radius:5px">

  <div style="background:rgb(218, 238, 132, 1);color:black;padding:0.9rem;font-weight:bold">Question 3.3</div>

- A **SMILES** string is a simple, one-line represention of a molecule.
  - On the PubChem page, can you identify the SMILES string for the compound? 
- On the CB-Dock 2 homepage, click on "Get started" to start the molecular docking process.
  - Upload the homology model that you generated and downloaded from SWISS-MODEL, and also the small molecule that you downloaded from PubChem. The small molecule will be moved around the protein's (receptor) surface computationally, to obtain a possible **docked poses** for the ligand, i.e. ones having **favourable binding energies** (lowest values).
  - How many putative ligand binding pockets were found on the protein?
  - What was the **energy score** of the best docked pose?
  - What are the protein residues to which the ligand was docked?
  - Hover over the protein-ligand contacts.
    - Which non-covalent interactions can you identify?
</div>

At the end of this exercise, you have effectively examined the propagation of a viral sequence, and evaluated a potentially inhibitory compound against a real protein from a pathogen, using **in silico** (i.e. on a computer) techniques. In a real life scenario, researchers from around the world carry out similar calculations on larger scales.

**Congratulations in reaching the end of this practical!**